In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Change paths below
FEATURES_INPUT_FILE = '/content/drive/MyDrive/2025_PPG_GLUC/Data/merged_clinical_lab_15_bins.csv'
OUTPUT_DIR = '/content/drive/MyDrive/2025_PPG_GLUC/Data/Model/100Hz/demographics_15_bin_concat'

# Split ratios
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Random seed
RANDOM_STATE = 42


def main():
    print("=" * 70)
    print("PPG Data Splitter - Case-Level Split")
    print("=" * 70)

    # Create output directory
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    # Load data
    print("\n1. Loading data...")
    df = new_merged
    print(f"   Total samples: {len(df):,}")
    print(f"   Total columns: {len(df.columns)}")
    print(f"   Unique cases: {df['caseid'].nunique()}")
    print(f"   Glucose range: [{df['glucose_value'].min():.1f}, {df['glucose_value'].max():.1f}] mg/dL")

    # CASE-LEVEL SPLIT
    print("\n2. Splitting data (case-level)...")

    groups = df['caseid'].values

    # First split: train vs (val + test)
    gss1 = GroupShuffleSplit(n_splits=1, test_size=(VAL_RATIO + TEST_RATIO), random_state=RANDOM_STATE)
    train_idx, temp_idx = next(gss1.split(df, groups=groups))

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_temp = df.iloc[temp_idx]

    # Second split: val vs test
    temp_groups = df_temp['caseid'].values
    val_ratio_adjusted = VAL_RATIO / (VAL_RATIO + TEST_RATIO)

    gss2 = GroupShuffleSplit(n_splits=1, test_size=(1 - val_ratio_adjusted), random_state=RANDOM_STATE)
    val_idx, test_idx = next(gss2.split(df_temp, groups=temp_groups))

    df_val = df_temp.iloc[val_idx].reset_index(drop=True)
    df_test = df_temp.iloc[test_idx].reset_index(drop=True)

    # ===== VERIFY NO LEAKAGE =====
    print("\n3. Verifying no data leakage...")

    train_cases = set(df_train['caseid'].unique())
    val_cases = set(df_val['caseid'].unique())
    test_cases = set(df_test['caseid'].unique())

    train_val_overlap = train_cases & val_cases
    train_test_overlap = train_cases & test_cases
    val_test_overlap = val_cases & test_cases

    print(f"   Train-Val overlap:  {len(train_val_overlap)} cases")
    print(f"   Train-Test overlap: {len(train_test_overlap)} cases")
    print(f"   Val-Test overlap:   {len(val_test_overlap)} cases")

    if len(train_val_overlap) == 0 and len(train_test_overlap) == 0 and len(val_test_overlap) == 0:
        print("   ✓ No data leakage detected!")
    else:
        print("   ✗ WARNING: Data leakage detected!")
        return

    # ===== SUMMARY =====
    print("\n4. Split Summary:")
    print(f"   {'Split':<10} {'Samples':>12} {'Percentage':>12} {'Cases':>10}")
    print("   " + "-" * 50)
    print(f"   {'Train':<10} {len(df_train):>12,} {100*len(df_train)/len(df):>11.1f}% {df_train['caseid'].nunique():>10}")
    print(f"   {'Val':<10} {len(df_val):>12,} {100*len(df_val)/len(df):>11.1f}% {df_val['caseid'].nunique():>10}")
    print(f"   {'Test':<10} {len(df_test):>12,} {100*len(df_test)/len(df):>11.1f}% {df_test['caseid'].nunique():>10}")
    print("   " + "-" * 50)
    print(f"   {'Total':<10} {len(df):>12,} {100.0:>11.1f}% {df['caseid'].nunique():>10}")

    # Glucose distribution per split
    print("\n   Glucose Distribution:")
    print(f"   {'Split':<10} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
    print("   " + "-" * 50)
    for name, split_df in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
        g = split_df['glucose_value']
        print(f"   {name:<10} {g.mean():>10.1f} {g.std():>10.1f} {g.min():>10.1f} {g.max():>10.1f}")

    # ===== SAVE =====
    print("\n5. Saving split data...")

    df_train.to_csv(f'{OUTPUT_DIR}/100Hz_train_15pulses_demographics.csv', index=False)
    df_val.to_csv(f'{OUTPUT_DIR}/100Hz_val_15pulses_demographics.csv', index=False)
    df_test.to_csv(f'{OUTPUT_DIR}/100Hz_test_15pulses_demographics.csv', index=False)

    print(f"   Saved: {OUTPUT_DIR}100Hz_train_15pulses.csv ({len(df_train):,} rows)")
    print(f"   Saved: {OUTPUT_DIR}100Hz_val_15pulses.csv ({len(df_val):,} rows)")
    print(f"   Saved: {OUTPUT_DIR}100Hz_test_15pulses.csv ({len(df_test):,} rows)")

    # Save split info
    split_info = pd.DataFrame({
        'split': ['train', 'val', 'test'],
        'samples': [len(df_train), len(df_val), len(df_test)],
        'cases': [df_train['caseid'].nunique(), df_val['caseid'].nunique(), df_test['caseid'].nunique()],
        'glucose_mean': [df_train['glucose_value'].mean(), df_val['glucose_value'].mean(), df_test['glucose_value'].mean()],
        'glucose_std': [df_train['glucose_value'].std(), df_val['glucose_value'].std(), df_test['glucose_value'].std()]
    })
    split_info.to_csv(f'{OUTPUT_DIR}/100Hz_15pulses_split_info_demographics.csv', index=False)
    print(f"   Saved: {OUTPUT_DIR}/100Hz_15pulses_split_info.csv")

    print("\n" + "=" * 70)
    print("Done!")
    print("=" * 70)


if __name__ == "__main__":
    main()